In [1]:
import os
import keras_hub
import keras

import tensorflow.data as tf_data
import tensorflow.strings as tf_strings

import gdown

2025-04-25 23:07:02.357989: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745622422.371198       9 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745622422.375126       9 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-25 23:07:02.389373: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [22]:
# Data
BATCH_SIZE = 1024
MIN_STRING_LEN = 300  # Strings shorter than this will be discarded
# SEQ_LEN = 128  # Length of training sequences, in tokens
SEQ_LEN = 128  # Length of training sequences, in tokens

# Model
EMBED_DIM = 256
FEED_FORWARD_DIM = 128
NUM_HEADS = 3
NUM_LAYERS = 2
VOCAB_SIZE = 5000  # Limits parameters in model.
# Training
EPOCHS = 5
# Inference
# NUM_TOKENS_TO_GENERATE = 80

In [3]:
dir = os.path.expanduser("data/fakenewsmakerGPT/")
nombreTextoTraining="texto_oraciones_Training_f2.txt"

url = "https://drive.google.com/uc?id=1gY9brunFfiLCB77GWJa9Usizw4qMSqIP"
try:
  os.mkdir(dir)
except FileExistsError:
    print(f"Directiorio data ya existe")
output = dir + nombreTextoTraining
gdown.download(url, output, quiet=False)

Directiorio data ya existe


Downloading...
From: https://drive.google.com/uc?id=1gY9brunFfiLCB77GWJa9Usizw4qMSqIP
To: /home/jupyter/data/fakenewsmakerGPT/texto_oraciones_Training_f2.txt
100%|██████████| 33.9M/33.9M [00:00<00:00, 61.0MB/s]


'data/fakenewsmakerGPT/texto_oraciones_Training_f2.txt'

In [23]:
# keras.utils.get_file(
#     origin="https://dldata-public.s3.us-east-2.amazonaws.com/simplebooks.zip",
#     extract=True,
# )
# dir = os.path.expanduser("~/.keras/datasets/simplebooks/")

# dir = os.path.expanduser("data/fakenewsmakerGPT/texto_oraciones_Training.txt")
# dir = os.path.expanduser("data/fakenewsmakerGPT/training20250421.txt")

# Load simplebooks-92 train set and filter out short lines.
raw_train_ds = (
    # tf_data.TextLineDataset(dir + "training20250421.txt")
    tf_data.TextLineDataset(dir + nombreTextoTraining)
    # tf_data.TextLineDataset(dir + "texto_oraciones_Training_f2.txt")
    .filter(lambda x: tf_strings.length(x) > MIN_STRING_LEN)
    .batch(BATCH_SIZE)
    .shuffle(buffer_size=256)
)


# raw_train_ds = (
#     # tf_data.TextLineDataset(dir + "training20250421.txt")
#     tf_data.TextLineDataset(dir + "texto_oraciones_Training.txt")
#     .filter(lambda x: tf_strings.length(x) > MIN_STRING_LEN)
#     .batch(BATCH_SIZE)
#     .shuffle(buffer_size=256)
# )


# # Load simplebooks-92 validation set and filter out short lines.
# raw_val_ds = (
#     tf_data.TextLineDataset(dir + "validation.txt")
#     .filter(lambda x: tf_strings.length(x) > MIN_STRING_LEN)
#     .batch(BATCH_SIZE)
# )

In [24]:
vocabDir=dir+"vocabNews.txt"

if not os.path.isfile(vocabDir):
    # Train tokenizer vocabulary
    vocab = keras_hub.tokenizers.compute_word_piece_vocabulary(
        raw_train_ds,
        vocabulary_size=VOCAB_SIZE,
        vocabulary_output_file=vocabDir,
        lowercase=True,
        reserved_tokens=["[PAD]", "[UNK]", "[BOS]"],
    )

tokenizer = keras_hub.tokenizers.WordPieceTokenizer(
    vocabulary=vocabDir,
    sequence_length=SEQ_LEN,
    lowercase=True,
)

# "trainfull.txt" 2min
# "training20250421.txt" 2min47


In [25]:
# packer adds a start token
start_packer = keras_hub.layers.StartEndPacker(
    sequence_length=SEQ_LEN,
    start_value=tokenizer.token_to_id("[BOS]"),
    # start_value=[tokenizer.token_to_id("la"),tokenizer.token_to_id("presidenta")],
    # start_value=[tokenizer.token_to_id("presidenta")],
    # end_value=".",
)
def preprocess(inputs):
    outputs = tokenizer(inputs)
    features = start_packer(outputs)
    labels = outputs
    return features, labels

In [26]:
# Tokenize and split into train and label sequences.
train_ds = raw_train_ds.map(preprocess, num_parallel_calls=tf_data.AUTOTUNE).prefetch(
    tf_data.AUTOTUNE
)
# val_ds = raw_val_ds.map(preprocess, num_parallel_calls=tf_data.AUTOTUNE).prefetch(
#     tf_data.AUTOTUNE
# )

In [27]:
inputs = keras.layers.Input(shape=(None,), dtype="int32")
# Embedding.
embedding_layer = keras_hub.layers.TokenAndPositionEmbedding(
    vocabulary_size=VOCAB_SIZE,
    sequence_length=SEQ_LEN,
    embedding_dim=EMBED_DIM,
    mask_zero=True,
)
x = embedding_layer(inputs)
# Transformer decoders.
for _ in range(NUM_LAYERS):
    decoder_layer = keras_hub.layers.TransformerDecoder(
        num_heads=NUM_HEADS,
        intermediate_dim=FEED_FORWARD_DIM,
    )
    x = decoder_layer(x)  # Giving one argument only skips cross-attention.
# Output.
outputs = keras.layers.Dense(VOCAB_SIZE)(x)
model = keras.Model(inputs=inputs, outputs=outputs)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
perplexity = keras_hub.metrics.Perplexity(from_logits=True, mask_token_id=0)
model.compile(optimizer="adam", loss=loss_fn, metrics=[perplexity])
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_3  │ (None, None, 256)      │     1,312,768 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_8           │ (None, None, 256)      │       329,085 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_9           │ (None, None, 256)      │       329,085 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, None, 5000)     │     1,285,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,255,938 (12.42 MB)

 Trainable params: 3,255,938 (12.42 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
# Training
EPOCHS = 300

model.fit(train_ds, epochs=EPOCHS)
# model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

#26seg - 1epochs 
# training20250421 1min 17seg



Epoch 1/300


W0000 00:00:1745630804.124435      73 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1745630804.347850      73 assert_op.cc:38] Ignoring Assert operator sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2025-04-26 01:26:47.557831: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_1', 56 bytes spill stores, 56 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_multiply_reduce_fusion', 28 bytes spill stores, 32 bytes spill loads



     10/Unknown 12s 363ms/step - loss: 7.9880 - perplexity: 3177.8149

W0000 00:00:1745630811.406671      72 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1745630811.541735      72 assert_op.cc:38] Ignoring Assert operator sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2025-04-26 01:26:52.453345: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_7', 24 bytes spill stores, 24 bytes spill loads

2025-04-26 01:26:52.678236: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_61', 568 bytes spill stores, 580 bytes spill loads

2025-04-26 01:26:53.036252: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 

23/23 ━━━━━━━━━━━━━━━━━━━━ 27s 791ms/step - loss: 7.4599 - perplexity: 2053.4202
Epoch 2/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 323ms/step - loss: 5.6674 - perplexity: 306.9265
Epoch 3/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 324ms/step - loss: 5.0941 - perplexity: 171.6299
Epoch 4/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 327ms/step - loss: 4.8439 - perplexity: 133.2405
Epoch 5/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 329ms/step - loss: 4.7086 - perplexity: 116.2003
Epoch 6/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 329ms/step - loss: 4.6292 - perplexity: 107.2438
Epoch 7/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 333ms/step - loss: 4.5299 - perplexity: 97.0228
Epoch 8/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 339ms/step - loss: 4.4694 - perplexity: 91.2345
Epoch 9/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 332ms/step - loss: 4.3746 - perplexity: 82.9351
Epoch 10/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 330ms/step - loss: 4.2972 - perplexity: 76.7006
Epoch 11/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 330ms/step - loss: 4.2410 - perplexity: 72.4619
Epoch

2025-04-26 01:34:08.971828: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 349662265196423199
2025-04-26 01:34:08.971887: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 17236628703095404961


23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 325ms/step - loss: 3.1236 - perplexity: 23.4558
Epoch 47/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 325ms/step - loss: 3.1009 - perplexity: 22.9219
Epoch 48/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 329ms/step - loss: 3.0783 - perplexity: 22.4013
Epoch 49/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 326ms/step - loss: 3.0634 - perplexity: 22.0651
Epoch 50/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 346ms/step - loss: 2.9477 - perplexity: 21.1360
Epoch 51/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 335ms/step - loss: 3.0503 - perplexity: 21.7741
Epoch 52/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 336ms/step - loss: 3.0460 - perplexity: 21.6791
Epoch 53/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 346ms/step - loss: 3.0300 - perplexity: 21.3313
Epoch 54/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 346ms/step - loss: 2.8816 - perplexity: 19.7680
Epoch 55/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 329ms/step - loss: 2.9840 - perplexity: 20.3689
Epoch 56/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 333ms/step - loss: 2.9853 - perplexity: 20.3943
Epoc

2025-04-26 01:35:54.917975: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]


23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 328ms/step - loss: 2.9939 - perplexity: 20.5638
Epoch 58/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 327ms/step - loss: 2.9702 - perplexity: 20.0801
Epoch 59/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 330ms/step - loss: 2.9255 - perplexity: 19.2050
Epoch 60/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 334ms/step - loss: 2.9197 - perplexity: 19.0872
Epoch 61/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 332ms/step - loss: 2.9105 - perplexity: 18.9095
Epoch 62/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 327ms/step - loss: 2.9080 - perplexity: 18.8634
Epoch 63/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 328ms/step - loss: 2.8975 - perplexity: 18.6626
Epoch 64/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 331ms/step - loss: 2.8806 - perplexity: 18.3467
Epoch 65/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 330ms/step - loss: 2.8798 - perplexity: 18.3293
Epoch 66/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 330ms/step - loss: 2.8567 - perplexity: 17.9099
Epoch 67/300
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 334ms/step - loss: 2.8514 - perplexity: 17.8162
E

In [35]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_3  │ (None, None, 256)      │     1,312,768 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_8           │ (None, None, 256)      │       329,085 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_9           │ (None, None, 256)      │       329,085 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, None, 5000)     │     1,285,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,767,816 (37.26 MB)

 Trainable params: 3,255,938 (12.42 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,511,878 (24.84 MB)

In [29]:
def next(prompt, cache, index):
    logits = model(prompt)[:, index - 1, :]
    # Ignore hidden states for now; only needed for contrastive search.
    hidden_states = None
    return logits, hidden_states, cache

# The "packer" layers adds the [BOS] token for us.
prompt_tokens = start_packer(tokenizer([""]))
prompt_tokens

<tf.Tensor: shape=(1, 128), dtype=int32, numpy=
array([[2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
      dtype=int32)>

In [ ]:
class TopKTextGenerator(keras.callbacks.Callback):
    """A callback to generate text from a trained model using top-k."""

    def __init__(self, k):
        self.sampler = keras_hub.samplers.TopKSampler(k)

    def on_epoch_end(self, epoch, logs=None):
        output_tokens = self.sampler(
            next=next,
            prompt=prompt_tokens,
            index=1,
        )
        txt = tokenizer.detokenize(output_tokens)
        print(f"Top-K search generated text: \n{txt}\n")


text_generation_callback = TopKTextGenerator(k=6)
# Dummy training loop to demonstrate callback.
# model.fit(train_ds.take(1), verbose=2, epochs=2, callbacks=[text_generation_callback])
model.fit(train_ds, verbose=1, epochs=200, callbacks=[text_generation_callback])
#200 epocas 175min Local training20250421

In [30]:
modeldir="./Playground/FakeNewsMakerBETA0.8_E300.keras"
model.save(modeldir)

In [31]:
modelloaded = keras.models.load_model(modeldir)

In [32]:
SEQ_LEN=128

# packer adds a start token
start_packer = keras_hub.layers.StartEndPacker(
    sequence_length=SEQ_LEN,
    start_value=tokenizer.token_to_id("[BOS]"),
    # start_value=tokenizer.token_to_id("presidenta"),
)

def preprocess(inputs):
    outputs = tokenizer(inputs)
    features = start_packer(outputs)
    labels = outputs
    return features, labels

def next2(prompt, cache, index):
    logits = modelloaded(prompt)[:, index - 1, :]
    # Ignore hidden states for now; only needed for contrastive search.
    hidden_states = None
    return logits, hidden_states, cache



In [34]:
entrada=""

# The "packer" layers adds the [BOS] token for us.
prompt_tokens = start_packer(tokenizer([entrada]))
prompt_tokens

print(f"Top K Sampler")
for i in range(1,11):
    for j in range(1,2):
        
        sampler = keras_hub.samplers.TopPSampler(p=i/10)
        output_tokens = sampler(
            next=next2,
            prompt=prompt_tokens,
            index=len(entrada.split())+1,
        )
        txt = tokenizer.detokenize(output_tokens)
        # print(f"k={i} \t{txt}")
        print(f"p={i/10} \t{txt}")

print(f"Top K Sampler")
for i in range(1,11):
    for j in range(1,2):
        sampler = keras_hub.samplers.TopKSampler(k=i)
        output_tokens = sampler(
            next=next2,
            prompt=prompt_tokens,
            index=len(entrada.split())+1,
        )
        txt = tokenizer.detokenize(output_tokens)
        print(f"k={i} \t{txt}")
        


Top K Sampler
p=0.1 	['[BOS] el gobierno de la ciudad de méxico se ha comunicador de tijuana , donde se prevé la cuota de la cotización de semanas y funcionarias de obras públicas se realizan tareas de rescate , explicó el impulso de la procuraduría en el sector , encabezado por el congreso estatal y el director del issste . [PAD] sitios . [PAD] circuitos culturales . [PAD]s culturales . [PAD] . [PAD]s , historia degráfico nacional , académicas y lenguas similares , trayectoria público social . [PAD] . [PAD] . [PAD] legado nacional carlos . [PAD] . [PAD] . [PAD] . [PAD] . [PAD] . [PAD]aián gber ofel máximos históricos que partidos oficialestanfe']
p=0.2 	['[BOS] la actitud del gobierno federal , que se convirtió en el presidente joe biden ayer figuras a la pandemia de covid - 19 , el pensamiento árt , interminatorias , instale opositores , chantip internationalen gusal , casualidad , cholula , dirigente de la asociación mexicana de del ejército y la guardia nacional ( gn ) , de la canc

In [ ]:
sampler = keras_hub.samplers.TopPSampler(p=0.75)
output_tokens = sampler(
    next=next2,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Top-P search generated text: \n{txt}\n")

In [ ]:
sampler = keras_hub.samplers.GreedySampler()
output_tokens = sampler(
    next=next2,
    prompt=prompt_tokens,
    index=1,  # Start sampling immediately after the [BOS] token.
)
txt = tokenizer.detokenize(output_tokens)
print(f"Greedy search generated text: \n{txt}\n")

In [ ]:
sampler = keras_hub.samplers.BeamSampler(num_beams=10)
output_tokens = sampler(
    next=next2,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Beam search generated text: \n{txt}\n")

In [ ]:
sampler = keras_hub.samplers.RandomSampler()
output_tokens = sampler(
    next=next2,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Random search generated text: \n{txt}\n")

In [ ]:
sampler = keras_hub.samplers.TopKSampler(k=6)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"",txt)